In [0]:
-- ============================================================
-- AUTO LOADER — Ingestão Incremental para Camada Bronze
-- ============================================================
-- Implementação futura para cenários com APIs ou ingestão contínua.
-- Baseado no padrão do notebook AutoLoader-Bronze.
--
-- Equivalência com o código de referência (Python):
--   spark.readStream.format("cloudFiles")        → STREAM read_files(...)
--   .option("cloudFiles.pathGlobFilter", "x.csv")  → WHERE _metadata.file_name LIKE 'x%'
--   .option("cloudFiles.schemaLocation", path)     → schemaLocation => path
--   .writeStream.option("checkpointLocation", p)   → gerenciado automaticamente pela streaming table
--   .trigger(availableNow=True)                   → CREATE OR REFRESH (idempotente, processa e para)
--   try/except (verifica se tabela existe)        → CREATE OR REFRESH (se já existe, só novos arquivos)
--
-- PREVENÇÃO "sem arquivos novos":
--   CREATE OR REFRESH STREAMING TABLE é idempotente:
--   - Primeira execução: carrega todos os arquivos do diretório.
--   - Execuções seguintes (REFRESH TABLE): processa apenas arquivos novos
--     desde o último checkpoint. Se nenhum arquivo novo chegou,
--     o REFRESH completa sem erro (0 linhas processadas).
--   - O checkpoint é gerenciado automaticamente pela streaming table.
--
-- ESCOPO B (hands-on): cada tabela Bronze deve conter data/hora de
-- ingestão (updated) E arquivo de origem (arquivo_origem).
--
-- MIGRAÇÃO: as tabelas bronze atuais são tabelas regulares (batch).
-- Para ativar o Auto Loader, remova-as primeiro (descomente abaixo na 1ª migração):
--   DROP TABLE IF EXISTS desafio_grupo1.bronze.bronze_clientes;
--   DROP TABLE IF EXISTS desafio_grupo1.bronze.bronze_vendas;
--   DROP TABLE IF EXISTS desafio_grupo1.bronze.bronze_suporte;
-- ============================================================

-- ============================================================
-- 1. AUTO LOADER — bronze_clientes
-- ============================================================
CREATE OR REFRESH STREAMING TABLE desafio_grupo1.bronze.bronze_clientes
COMMENT 'Auto Loader — ingestão incremental de clientes via CSV'
AS SELECT
  CAST(id_cliente AS INT) AS id_cliente,
  nome,
  sexo,
  CAST(data_nascimento AS DATE) AS data_nascimento,
  cidade,
  estado,
  CAST(data_cadastro AS DATE) AS data_cadastro,
  _metadata.file_name AS arquivo_origem,
  CURRENT_TIMESTAMP() AS updated
FROM STREAM read_files(
  '/Volumes/desafio_grupo1/landing/files/',
  format => 'csv',
  header => 'true',
  schemaLocation => '/Volumes/desafio_grupo1/landing/metadata/clientes/schema'
)
WHERE _metadata.file_name LIKE 'clientes%';

-- ============================================================
-- 2. AUTO LOADER — bronze_vendas
-- ============================================================
CREATE OR REFRESH STREAMING TABLE desafio_grupo1.bronze.bronze_vendas
COMMENT 'Auto Loader — ingestão incremental de vendas via CSV'
AS SELECT
  CAST(id_venda AS INT) AS id_venda,
  CAST(id_cliente AS INT) AS id_cliente,
  CAST(data_venda AS DATE) AS data_venda,
  produto,
  CAST(quantidade AS INT) AS quantidade,
  CAST(valor_total AS DECIMAL(10,2)) AS valor_total,
  _metadata.file_name AS arquivo_origem,
  CURRENT_TIMESTAMP() AS updated
FROM STREAM read_files(
  '/Volumes/desafio_grupo1/landing/files/',
  format => 'csv',
  header => 'true',
  schemaLocation => '/Volumes/desafio_grupo1/landing/metadata/vendas/schema'
)
WHERE _metadata.file_name LIKE 'vendas%';

-- ============================================================
-- 3. AUTO LOADER — bronze_suporte
-- ============================================================
CREATE OR REFRESH STREAMING TABLE desafio_grupo1.bronze.bronze_suporte
COMMENT 'Auto Loader — ingestão incremental de suporte via CSV'
AS SELECT
  CAST(id_interacao AS INT) AS id_interacao,
  CAST(id_cliente AS INT) AS id_cliente,
  canal,
  tipo_problema,
  CAST(tempo_resolucao AS INT) AS tempo_resolucao,
  CAST(satisfacao_cliente AS INT) AS satisfacao_cliente,
  _metadata.file_name AS arquivo_origem,
  CURRENT_TIMESTAMP() AS updated
FROM STREAM read_files(
  '/Volumes/desafio_grupo1/landing/files/',
  format => 'csv',
  header => 'true',
  schemaLocation => '/Volumes/desafio_grupo1/landing/metadata/suporte/schema'
)
WHERE _metadata.file_name LIKE 'suporte%';

-- ============================================================
-- REFRESH INCREMENTAL — executar em runs subsequentes do Job
-- ============================================================
-- Processa apenas arquivos novos desde o último checkpoint.
-- Se nenhum arquivo novo chegou, cada REFRESH completa sem erro (0 linhas).
-- Equivalente ao trigger(availableNow=True) do código de referência.
--
-- REFRESH TABLE desafio_grupo1.bronze.bronze_clientes;
-- REFRESH TABLE desafio_grupo1.bronze.bronze_vendas;
-- REFRESH TABLE desafio_grupo1.bronze.bronze_suporte;

-- ============================================================
-- VERIFICAÇÃO
-- ============================================================
SELECT 'bronze_clientes' AS tabela, COUNT(*) AS total FROM desafio_grupo1.bronze.bronze_clientes
UNION ALL
SELECT 'bronze_vendas', COUNT(*) FROM desafio_grupo1.bronze.bronze_vendas
UNION ALL
SELECT 'bronze_suporte', COUNT(*) FROM desafio_grupo1.bronze.bronze_suporte;